# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [ ]:
# Write your code below.
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")

parquet_files = glob(
    os.path.join(PRICE_DATA, "**", "*.parquet"),
    recursive=True
)

print("PRICE_DATA:", PRICE_DATA)
print("Number of parquet files:", len(parquet_files))

parquet_files[:5]

PRICE_DATA: ../../05_src/data/prices/
Number of parquet files: 3146


['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet']

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.
# Only load the columns needed for this feature creation task
dd_price = dd.read_parquet(
    parquet_files,
    columns=["Date", "High", "Low", "Close", "Adj Close", "ticker"]
)

# Rename Adj Close 
dd_price = dd_price.rename(columns={"Adj Close": "Adj_Close"})

# Define a function that will be applied to each Dask partition
def add_features(df):
    df = df.sort_values(["ticker", "Date"]).reset_index(drop=True)

    # Step 1 Add lag features for Close and Adj_Close
    df["Close_lag_1"] = df.groupby("ticker")["Close"].shift(1)
    df["Adj_Close_lag_1"] = df.groupby("ticker")["Adj_Close"].shift(1)

    # Step 2 Add returns based on Close
    df["returns"] = (df["Close"] / df["Close_lag_1"]) - 1

    # Step 3 Add high-low range
    df["hi_lo_range"] = df["High"] - df["Low"]

    return df

meta = dd_price._meta.copy()
meta["Close_lag_1"] = "float64"
meta["Adj_Close_lag_1"] = "float64"
meta["returns"] = "float64"
meta["hi_lo_range"] = "float64"

# Step 4 Assign the result to dd_feat
dd_feat = dd_price.map_partitions(add_features, meta=meta)

dd_feat.columns

Index(['Date', 'High', 'Low', 'Close', 'Adj_Close', 'ticker', 'Close_lag_1',
       'Adj_Close_lag_1', 'returns', 'hi_lo_range'],
      dtype='object')

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [51]:
# Write your code below.
# Convert the Dask dataframe to a pandas dataframe
df_feat = dd_feat.compute()

# Sort by ticker and Date before calculating rolling average
df_feat = df_feat.sort_values(["ticker", "Date"]).reset_index(drop=True)

# Add 10-day moving average of returns for each ticker
df_feat["returns_ma_10"] = (
    df_feat
    .groupby("ticker")["returns"]
    .transform(lambda s: s.rolling(10).mean())
)

df_feat.head()

,Date,High,Low,Close,Adj_Close,ticker,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
0,2001-07-19,15.29,15.00,15.17,11.404394,ACN,NaN,NaN,NaN,0.29,NaN
1,2001-07-20,15.05,14.80,15.01,11.284108,ACN,15.17,11.404394,-0.010547,0.25,NaN
2,2001-07-23,15.01,14.55,15.00,11.276587,ACN,15.01,11.284108,-0.000666,0.46,NaN
3,2001-07-24,14.97,14.70,14.86,11.171341,ACN,15.00,11.276587,-0.009333,0.27,NaN
4,2001-07-25,14.95,14.65,14.95,11.238999,ACN,14.86,11.171341,0.006057,0.30,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

It was not strictly necessary to convert to pandas because Dask can also perform rolling calculations. However, using pandas was acceptable here because the data could fit into memory and the rolling average was simple to implement.

+ Would it have been better to do it in Dask? Why?

For a larger production dataset, it would generally be better to keep the calculation in Dask because Dask can process partitioned parquet files without loading the entire dataset into memory at once. This is more scalable and more consistent with the purpose of using Dask and parquet for larger data processing workflows.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.